# 01 · Setup and test pull

**Phase 1 check:** confirm the data source works before the full collection.

Data comes from the [Arctic Shift](https://github.com/ArthurHeitmann/arctic_shift) public Reddit archive, not Reddit's official API, so no credentials or `.env` file are needed. Usernames are never requested in this notebook.

*Independent student project, not affiliated with WHOOP.*

In [3]:
import requests
import pandas as pd
from datetime import datetime, timezone

BASE = "https://arctic-shift.photon-reddit.com/api"
HEADERS = {"User-Agent": "whoop-emotion-student-project/0.2 (non-commercial research)"}
START, END = "2025-09-28", "2026-09-28"
SUBREDDITS = {"WHOOP": "whoop", "Oura": "ouraring", "Garmin": "Garmin", "Apple Watch": "AppleWatch"}

def search_posts(sub, limit=10, sort="desc", **extra):
    params = {"subreddit": sub, "after": START, "before": END, "limit": limit, "sort": sort,
              "fields": "id,created_utc,score,num_comments,title", **extra}
    r = requests.get(f"{BASE}/posts/search", params=params, headers=HEADERS, timeout=60)
    r.raise_for_status()
    payload = r.json()
    if payload.get("error"):
        raise RuntimeError(payload["error"])
    return pd.DataFrame(payload["data"])

def utc_date(ts):
    return datetime.fromtimestamp(int(ts), tz=timezone.utc).strftime("%Y-%m-%d")

## Test pull: 10 posts from r/whoop

In [4]:
test = search_posts("whoop", limit=10)
test["date"] = test["created_utc"].map(utc_date)
print(f"Pulled {len(test)} posts from r/whoop")
test[["date", "score", "num_comments", "title"]]

Pulled 10 posts from r/whoop


,date,score,num_comments,title
0,2026-09-27,0,11,EMF from whoop
1,2026-09-27,0,14,5.0 battery low only 8 days in
2,2026-09-27,32,9,Appreciation.
3,2026-09-27,0,1,Aiutatemi a decidere
4,2026-09-27,0,9,Whoop burned me
5,2026-09-27,4,7,Advice on what I can do better!
6,2026-09-27,13,16,Lean body mass and whoop age
7,2026-09-27,5,10,4.0 vs 5.0 - 5.0 seems inaccurate
8,2026-09-27,17,3,First week of Whoop
9,2026-09-27,1,1,looking to buy whoop 5.0/MG


## Coverage check: does each subreddit have data across the whole window?

Pulls the oldest and newest post in the window for each brand. Both dates should sit near the window edges (Sep 28, 2025 and Sep 28, 2026).

In [5]:
rows = []
for brand, sub in SUBREDDITS.items():
    oldest = search_posts(sub, limit=1, sort="asc")
    newest = search_posts(sub, limit=1, sort="desc")
    rows.append({
        "brand": brand,
        "subreddit": f"r/{sub}",
        "oldest_post_in_window": utc_date(oldest.created_utc[0]) if len(oldest) else None,
        "newest_post_in_window": utc_date(newest.created_utc[0]) if len(newest) else None,
    })
coverage = pd.DataFrame(rows)
coverage

,brand,subreddit,oldest_post_in_window,newest_post_in_window
0,WHOOP,r/whoop,2025-09-28,2026-09-27
1,Oura,r/ouraring,2025-09-28,2026-09-27
2,Garmin,r/Garmin,2025-09-28,2026-09-27
3,Apple Watch,r/AppleWatch,2025-09-28,2026-09-27


## Phase 1 done when

- [x] Test pull returns 10 r/whoop posts
- [x] All four subreddits have posts at both ends of the window
- [ ] Problem statement and hypotheses written: `docs/problem_statement.md`
- [ ] GitHub repo created with README and `.gitignore` (excludes `.env` and `data/`)

Next: run `python src/collect_reddit.py --quick`, then the full `python src/collect_reddit.py`.